# Localized Tensors

This notebook runs a cuDNN graph on tensors whose memory sits in one **locality domain** of the GPU.

Some GPUs, Rubin among them, divide their SMs and memory into locality domains. The CUDA virtual memory management (VMM) API can place an allocation in one chosen domain. cuDNN needs no special support for this. You allocate the memory yourself and pass cuDNN an ordinary device pointer, and the graph never learns that the memory was localized.

That split is the point of this notebook: **everything locality-specific happens in the allocator, above cuDNN.** From `graph.validate()` to `graph.execute()`, this is an ordinary cuDNN Frontend program.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NVIDIA/cudnn-frontend/blob/main/samples/python/03_localized_tensors.ipynb)

## Prerequisites

- **A GPU that reports at least two locality domains.** With one domain, a localized allocation is the same as an ordinary full-device allocation, so there is nothing to show. The notebook skips in that case.
- **CUDA 13.4 or newer**, for both the driver and `cuda-bindings` (the Python driver bindings). `CU_MEM_LOCATION_TYPE_DEVICE_LOCALITY_DOMAIN` first appears in CUDA 13.4.
- **PyTorch**, used here only to create the inputs and compute a reference result.

In [ ]:
# get_ipython().system('nvidia-smi')

If running on Colab, you will need to install the cudnn python interface and the CUDA driver bindings.

In [ ]:
# get_ipython().system('pip install nvidia-cudnn-cu13')
# get_ipython().system('pip install nvidia-cudnn-frontend')
# get_ipython().system("pip install 'cuda-bindings>=13.4'")
# get_ipython().system('pip3 install torch --index-url https://download.pytorch.org/whl/cu130')

## Checking the device

The next cell decides whether this notebook can run on your system. It checks that:

1. The installed `cuda-bindings` defines `CU_MEM_LOCATION_TYPE_DEVICE_LOCALITY_DOMAIN`. The check looks for the enum itself, not a version string.
2. The CUDA driver is 13.4 or newer.
3. The device reports at least two locality domains.

If a check fails, the cell prints `SKIP:` with the reason, and every later cell that needs locality domains prints `Skipping ...` instead of running.

Otherwise it makes the device's primary context current. That is the context PyTorch and cuDNN use, and the driver calls below need one.

In [ ]:
import cudnn
import torch
from cuda.bindings import driver as cu


def check(result):
    """cuda-bindings returns (error, *values). Raise on error, otherwise return the values."""
    err, *values = result if isinstance(result, tuple) else (result,)
    if err != cu.CUresult.CUDA_SUCCESS:
        raise RuntimeError(f"CUDA driver error: {err}")
    if not values:
        return None
    return values[0] if len(values) == 1 else tuple(values)


def device_attribute(attribute):
    return check(cu.cuDeviceGetAttribute(attribute, cu_device))


device = 0
skip_reason = None

if not hasattr(cu.CUmemLocationType, "CU_MEM_LOCATION_TYPE_DEVICE_LOCALITY_DOMAIN"):
    skip_reason = "this cuda-bindings has no CU_MEM_LOCATION_TYPE_DEVICE_LOCALITY_DOMAIN; install cuda-bindings >= 13.4"
else:
    check(cu.cuInit(0))
    driver_version = check(cu.cuDriverGetVersion())
    if driver_version < 13040:
        skip_reason = f"locality domains need a CUDA 13.4 or newer driver; this driver is {driver_version}"

if skip_reason is None:
    cu_device = check(cu.cuDeviceGet(device))
    num_domains = device_attribute(cu.CUdevice_attribute.CU_DEVICE_ATTRIBUTE_LOCALITY_DOMAIN_COUNT)
    sms_per_domain = device_attribute(cu.CUdevice_attribute.CU_DEVICE_ATTRIBUTE_LOCALITY_DOMAIN_MULTIPROCESSOR_COUNT)
    sms_total = device_attribute(cu.CUdevice_attribute.CU_DEVICE_ATTRIBUTE_MULTIPROCESSOR_COUNT)

    print(f"device {device}: {torch.cuda.get_device_name(device)}")
    print(f"locality domains: {num_domains}, SMs per domain: {sms_per_domain}, SMs on the device: {sms_total}")
    print(f"cuDNN backend version: {cudnn.backend_version()}")

    if num_domains < 2:
        skip_reason = f"device {device} reports {num_domains} locality domain(s), so a localized allocation would be an ordinary full-device one"
    else:
        check(cu.cuCtxSetCurrent(check(cu.cuDevicePrimaryCtxRetain(cu_device))))

has_locality_domains = skip_reason is None
if not has_locality_domains:
    print(f"SKIP: {skip_reason}")

Read the SM counts for your own device rather than assuming a split. Each domain holds fewer than half the SMs, and the exact count varies by board. The domains together do not cover every SM, so a domain is **not** "half the SMs".

## Allocating localized memory

A plain `cudaMalloc` cannot choose a domain. The VMM API can, in four calls:

1. `cuMemCreate` creates the physical memory. **This is the only locality-specific call.** Its location type is `CU_MEM_LOCATION_TYPE_DEVICE_LOCALITY_DOMAIN`, and the location names both the device and the domain.
2. `cuMemAddressReserve` reserves a range of virtual addresses.
3. `cuMemMap` maps the memory into that range.
4. `cuMemSetAccess` lets the device read and write the range. Access is granted to the **device** (`CU_MEM_LOCATION_TYPE_DEVICE`), whichever domain holds the bytes.

One trap to avoid: `CUmemLocation.id` and `CUmemLocation.localized.{deviceId, localityDomainId}` share the same bytes. Set `location.type` first, then write only the `localized` fields. If you read `id` back instead, you get a plausible but wrong number, not an error.

Freeing undoes the four calls in reverse order. The class below is a context manager: leaving its `with` block waits for pending GPU work and then frees the memory, also when an error is raised. If one of the four calls fails, the constructor undoes the calls that succeeded before it raises.

In [ ]:
class LocalizedBuffer:
    """Device memory placed in one locality domain. Use it in a `with` block, which frees the memory on exit."""

    def __init__(self, nbytes, device, domain):
        prop = cu.CUmemAllocationProp()
        prop.type = cu.CUmemAllocationType.CU_MEM_ALLOCATION_TYPE_PINNED
        prop.requestedHandleTypes = cu.CUmemAllocationHandleType.CU_MEM_HANDLE_TYPE_NONE
        # The location type first, then the union fields that go with it.
        prop.location.type = cu.CUmemLocationType.CU_MEM_LOCATION_TYPE_DEVICE_LOCALITY_DOMAIN
        prop.location.localized.deviceId = device
        prop.location.localized.localityDomainId = domain

        granularity = check(cu.cuMemGetAllocationGranularity(prop, cu.CUmemAllocationGranularity_flags.CU_MEM_ALLOC_GRANULARITY_MINIMUM))
        self.size = -(-max(nbytes, 1) // granularity) * granularity  # round up to the granularity
        self.handle = self.ptr = None
        self.mapped = False
        try:
            self.handle = check(cu.cuMemCreate(self.size, prop, 0))
            self.ptr = check(cu.cuMemAddressReserve(self.size, granularity, 0, 0))
            check(cu.cuMemMap(self.ptr, self.size, 0, self.handle, 0))
            self.mapped = True

            access = cu.CUmemAccessDesc()
            access.location.type = cu.CUmemLocationType.CU_MEM_LOCATION_TYPE_DEVICE
            access.location.id = device
            access.flags = cu.CUmemAccess_flags.CU_MEM_ACCESS_FLAGS_PROT_READWRITE
            check(cu.cuMemSetAccess(self.ptr, self.size, [access], 1))
        except BaseException:
            self.close()  # undo the calls that succeeded, then raise the original error
            raise

    def __int__(self):
        """The device address, which is all cuDNN needs."""
        return int(self.ptr)

    def __enter__(self):
        return self

    def __exit__(self, *exc_info):
        self.close()

    def close(self):
        """Wait for pending GPU work, then undo each allocation step that succeeded. Calling it again does nothing."""
        # Errors are not raised here, so freeing never hides the error that led to it.
        cu.cuCtxSynchronize()
        if self.mapped:
            cu.cuMemUnmap(self.ptr, self.size)
            self.mapped = False
        if self.ptr is not None:
            cu.cuMemAddressFree(self.ptr, self.size)
            self.ptr = None
        if self.handle is not None:
            cu.cuMemRelease(self.handle)
            self.handle = None

### Checking where the memory landed

Ask the driver where the memory is, rather than trusting the request. The pointer attribute `CU_POINTER_ATTRIBUTE_LOCALITY_DOMAIN_ORDINAL` returns the domain that holds an address, or -1 when the memory is not localized. Below, one buffer goes into each domain, and an ordinary PyTorch allocation serves as the contrast.

In [ ]:
def locality_domain_of(ptr):
    """The locality domain holding device address `ptr`, or -1 if the memory is not localized."""
    return check(cu.cuPointerGetAttribute(cu.CUpointer_attribute.CU_POINTER_ATTRIBUTE_LOCALITY_DOMAIN_ORDINAL, ptr))


if has_locality_domains:
    for domain in range(num_domains):
        with LocalizedBuffer(1 << 20, device, domain) as buf:
            placed = locality_domain_of(int(buf))
        print(f"requested domain {domain}: driver reports domain {placed}")
        assert placed == domain

    ordinary = torch.empty(1 << 20, dtype=torch.uint8, device="cuda")
    print(f"ordinary PyTorch allocation: driver reports {locality_domain_of(ordinary.data_ptr())}")
    del ordinary
else:
    print(f"Skipping the placement check ({skip_reason})")

## Building the graph

The graph is built the usual way, with one difference: it does not use `tensor_like()`. A raw pointer carries no shape, so the graph declares each tensor's dimensions, strides and data type itself. When `execute()` receives a bare integer address, the frontend reads it with the geometry the graph declared.

Nothing in the graph mentions a locality domain. One graph therefore serves every domain; only the pointers passed to `execute()` change.

In [ ]:
if has_locality_domains:
    handle = cudnn.create_handle()
    stream = torch.cuda.current_stream()
    cudnn.set_stream(handle=handle, stream=stream.cuda_stream)

    B, M, N, K = 16, 32, 64, 128
    dtype = torch.bfloat16

    # The inputs and the reference result live in ordinary PyTorch memory.
    a_ref = torch.randn(B, M, K, device="cuda", dtype=dtype)
    b_ref = torch.randn(B, K, N, device="cuda", dtype=dtype)
    c_ref = torch.matmul(a_ref, b_ref)

    graph = cudnn.pygraph(handle=handle)
    a = graph.tensor(name="A", dim=[B, M, K], stride=[M * K, K, 1], data_type=cudnn.data_type.BFLOAT16)
    b = graph.tensor(name="B", dim=[B, K, N], stride=[K * N, N, 1], data_type=cudnn.data_type.BFLOAT16)
    c = graph.matmul(name="matmul", A=a, B=b, compute_data_type=cudnn.data_type.FLOAT)
    c.set_output(True).set_data_type(cudnn.data_type.BFLOAT16)

    graph.validate()
    graph.build_operation_graph()
    graph.create_execution_plans([cudnn.heur_mode.A, cudnn.heur_mode.FALLBACK])
    graph.check_support()
    graph.build_plans()
    print(f"workspace: {graph.get_workspace_size()} bytes")
else:
    print(f"Skipping the graph build ({skip_reason})")

## Executing in each domain

For each domain, the next cell:

1. Allocates the operands and the workspace in that domain, and checks where the driver placed them.
2. Copies the inputs in.
3. Executes the graph, passing plain integers in the variant pack.
4. Frees the buffers when the `with` block ends, after the pending work has finished.
5. Compares the result with the PyTorch reference.

The copies are enqueued on the stream cuDNN runs on, so they are ordered with the matmul without any extra synchronization.

**Only the tensors need localizing.** The workspace is scratch memory, and an ordinary allocation works just as well. It is localized here only for consistency.

In [ ]:
def nbytes(t):
    return t.numel() * t.element_size()


def run_in_domain(domain):
    # Leaving the `with` block frees the buffers, also on an error.
    with (
        LocalizedBuffer(nbytes(a_ref), device, domain) as a_buf,
        LocalizedBuffer(nbytes(b_ref), device, domain) as b_buf,
        LocalizedBuffer(nbytes(c_ref), device, domain) as c_buf,
        # Optional: the workspace works from ordinary memory too. Localized here for consistency.
        LocalizedBuffer(graph.get_workspace_size(), device, domain) as workspace,
    ):
        for buf in (a_buf, b_buf, c_buf, workspace):
            placed = locality_domain_of(int(buf))
            assert placed == domain, f"asked for domain {domain}, the driver placed the buffer in {placed}"

        check(cu.cuMemcpyDtoDAsync(int(a_buf), a_ref.data_ptr(), nbytes(a_ref), stream.cuda_stream))
        check(cu.cuMemcpyDtoDAsync(int(b_buf), b_ref.data_ptr(), nbytes(b_ref), stream.cuda_stream))

        # Plain integers: to cuDNN these are ordinary device pointers.
        graph.execute({a: int(a_buf), b: int(b_buf), c: int(c_buf)}, workspace=int(workspace), handle=handle)

        c_out = torch.empty_like(c_ref)
        check(cu.cuMemcpyDtoDAsync(c_out.data_ptr(), int(c_buf), nbytes(c_out), stream.cuda_stream))

    torch.testing.assert_close(c_out, c_ref, rtol=2e-2, atol=2e-2)
    print(f"domain {domain}: matmul on localized tensors matches the reference")


if has_locality_domains:
    for domain in range(num_domains):
        run_in_domain(domain)
else:
    print(f"Skipping the matmul in each domain ({skip_reason})")

## Cleanup

In [ ]:
if has_locality_domains:
    cudnn.destroy_handle(handle)
    check(cu.cuDevicePrimaryCtxRelease(cu_device))
else:
    print(f"Skipping cleanup ({skip_reason})")

## Summary

- Locality is chosen when the memory is allocated: `cuMemCreate` with `CU_MEM_LOCATION_TYPE_DEVICE_LOCALITY_DOMAIN`. Access is still granted to the device.
- `CU_POINTER_ATTRIBUTE_LOCALITY_DOMAIN_ORDINAL` confirms the placement in one call, and returns -1 for memory that is not localized.
- cuDNN receives plain device pointers. The graph was built once and ran unchanged in every domain, because nothing from `validate()` to `execute()` refers to locality.
- Only the tensors need localizing. The workspace can come from anywhere.